In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/

/content/drive/MyDrive


In [3]:
!pip -q install scipy

# 라이브러리

In [4]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
from typing import Dict, Tuple

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# 시각화
import matplotlib.pyplot as plt

# 텍스트 데이터 전처리
import re
# import contractions
from collections import Counter
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize

# 그래프 생성
import scipy.sparse as sp

# Tensorflow 라이브러리
import tensorflow as tf
tf.config.optimizer.set_jit(False)


# Tensorflow 모델 구조
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization,  Lambda, ReLU, Softmax
from tensorflow.keras.layers import Input, Embedding, Flatten, Layer, Multiply, Concatenate, Dropout, Dense, Reshape, Permute, Dot
from tensorflow.keras.utils import plot_model
from tensorflow.keras.initializers import GlorotUniform

# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils


In [5]:
def build_tf_sparse_adj(user_train, item_train, n_users, n_items):
    data = np.ones_like(user_train, dtype=np.float32)
    R_train = sp.coo_matrix((data, (user_train, item_train)), shape=(n_users, n_items))

    # 2) 무향 인접행렬 A 구성
    A = sp.bmat([[None,       R_train],
                 [R_train.T, None      ]], format='coo')

    # 3) Degree 행렬 D^-1/2 계산
    deg = np.array(A.sum(axis=1)).flatten()
    deg_inv_sqrt = np.power(deg, -0.5, where=deg>0)
    D_inv_sqrt = sp.diags(deg_inv_sqrt)

    # 4) 대칭 정규화: Ã = D^-1/2 · A · D^-1/2
    A_norm = D_inv_sqrt.dot(A).dot(D_inv_sqrt).tocoo()

    # 5) TensorFlow 희소텐서로 변환 (PyTorch 대신)
    indices = np.stack([A_norm.row.astype(np.int64),
                        A_norm.col.astype(np.int64)], axis=1)
    values = A_norm.data.astype(np.float32)
    shape = A_norm.shape
    adj_norm_tf = tf.SparseTensor(indices=indices, values=values, dense_shape=shape)
    adj_norm_tf = tf.sparse.reorder(adj_norm_tf)
    return adj_norm_tf

In [6]:
def build_modal_projectors(d_text_raw, d_vis_raw, d):

    # text embedding
    proj_text = Sequential([Input(shape=(d_text_raw,)), Dense(d, use_bias=True)], name="proj_text")

    # visual embedding
    proj_vis = Sequential([Input(shape=(d_vis_raw,)), Dense(d, use_bias=True)], name="proj_vis")

    return proj_text, proj_vis

In [7]:
class GraphProp(Layer):
    def __init__(self, A_norm_tf: tf.SparseTensor, activation=None, dropout_rate=0.0, use_residual=False, **kwargs):
        super().__init__(**kwargs)
        self.A = tf.sparse.reorder(A_norm_tf)
        self.act = activation
        self.do = Dropout(dropout_rate) if dropout_rate and dropout_rate > 0 else None
        self.use_residual = use_residual

    def call(self, X, training=False):
        X_in = self.do(X, training=training) if self.do is not None else X
        X_prop = tf.sparse.sparse_dense_matmul(self.A, X_in)
        if self.use_residual:
            X_prop = X_prop + X
        if self.act is not None:
            X_prop = self.act(X_prop)
        return X_prop

In [8]:
# 2) 이중모달 어텐션(모달 수준 attention) 레이어
#    - 두 모달 임베딩 t, v (각 R^d)를 받아 모달 간 중요도 α를 산출하고
#      α_t * t + α_v * v 로 융합 표현을 만든 뒤, 원본과 concat

class BiModalAttention(Layer):
    def __init__(self, d, **kwargs):
        super().__init__(**kwargs)

        # 공용 쿼리 벡터 w: 각 모달의 "주목도"를 1스칼라로 산출
        self.w = self.add_weight(name="w_att", shape=(d, 1), initializer="glorot_uniform", trainable=True)

    def call(self, t, v):
        # s_t = t @ w, s_v = v @ w   -> (N,1)
        s_t = tf.linalg.matmul(t, self.w)
        s_v = tf.linalg.matmul(v, self.w)
        logits = tf.concat([s_t, s_v], axis=-1)           # (N, 2)
        alpha = tf.nn.softmax(logits, axis=-1)            # (N, 2)
        alpha_t = alpha[:, 0:1]
        alpha_v = alpha[:, 1:2]
        fused = alpha_t * t + alpha_v * v                 # (N, d)
        # 최종: [fused, t, v] 를 반환하여 풍부한 표현 제공
        return tf.concat([fused, t, v], axis=-1)          # (N, 3d)

In [9]:
# 3) 아이템 모달 인코더 (Functional)
#    - 제공된 projector로 텍스트/비주얼을 각각 d로 선형 투영
#    - BiModalAttention으로 융합 후, 최종 선형 사상(Dense(d, bias=False))

def build_item_encoder_with_bimodal_attention(d_text_raw, d_vis_raw, d):
    proj_text, proj_vis = build_modal_projectors(d_text_raw, d_vis_raw, d)

    t_in = Input(shape=(d_text_raw,), name="item_text_raw")
    v_in = Input(shape=(d_vis_raw,),  name="item_vis_raw")

    t_proj = proj_text(t_in)  # (N?, d)
    v_proj = proj_vis(v_in)   # (N?, d)

    bim_att = BiModalAttention(d, name="bimodal_attn")
    tv_fused = bim_att(t_proj, v_proj)                    # (N?, 3d)

    # 논문 취지: 어텐션 기반 융합 후 차원 정렬만 선형 사상으로 수행
    item_embed = Dense(d, use_bias=False, name="item_linear_out")(tv_fused)

    return Model([t_in, v_in], item_embed, name="ItemEncoder_BiAttn")

# 데이터셋 로딩

In [11]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_v5_final.pkl')
print(df.shape)
df.head()

(23491, 17)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb,user_bert_emb,item_bert_emb,user_roberta_emb,item_roberta_emb,LLaMA_Output,LLaVA_Output
0,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0BJKYJR3L,B0BJKYJR3L,"Nice privacy shield. Good color, fits well, et...",3,Jephuwneh 2 Pack Gradient Tempered Glass for i...,"[-0.0753104165, 0.1205661818, -0.0582872108, -...",https://m.media-amazon.com/images/I/515oPLx13k...,https://m.media-amazon.com/images/I/515oPLx13k...,https://m.media-amazon.com/images/I/515oPLx13k...,"[9.4065265656, 28.670337677, 22.95885849, 26.4...","[-0.50728655, -0.22111754, 0.44898894, -0.2970...","[-0.3555999, -0.19625245, 0.338061, -0.1889423...","[-0.053886086, 0.1409396, -0.025606845, -0.112...","[-0.05482221, 0.11866534, -0.037660137, -0.100...","Based on this user review, it appears that the...",The image shows a cellphone with a screen prot...
1,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0BHTZQHQW,B0BHTZQHQW,"This thing is great. It's not too big, has a d...",5,"NEOCOR Magnetic Wireless Power Bank, 10,000mAh...","[-0.1076164767, 0.15895271300000002, -0.038697...",https://m.media-amazon.com/images/I/71K8BDxzsA...,https://m.media-amazon.com/images/I/71K8BDxzsA...,https://m.media-amazon.com/images/I/71K8BDxzsA...,"[15.0432891846, 40.8174591064, 0.0, 27.5127182...","[-0.50728655, -0.22111754, 0.44898894, -0.2970...","[-0.5915019, -0.69345254, 0.21792011, -0.13003...","[-0.053886086, 0.1409396, -0.025606845, -0.112...","[-0.07991053, 0.16881612, -0.014415792, -0.127...","Based on the user review, here is a summary of...",The image shows a wireless power bank with a m...
2,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0B9N4VRB9,B0BHSSCXB2,This is a much better way to mount my iPhone (...,5,Lamicall Gooseneck Phone Holder Bed - [Newest ...,"[-0.098338306, 0.1566942036, -0.0378348567, -0...",https://m.media-amazon.com/images/I/71b+Sx-Jq7...,https://m.media-amazon.com/images/I/71b+Sx-Jq7...,https://m.media-amazon.com/images/I/71b+Sx-Jq7...,"[0.0, 57.4749221802, 14.7353601456, 36.8316497...","[-0.50728655, -0.22111754, 0.44898894, -0.2970...","[-0.11259819, -0.25781307, 0.46874186, -0.3961...","[-0.053886086, 0.1409396, -0.025606845, -0.112...","[-0.085501574, 0.112952866, -0.022212008, -0.0...","Based on the user review, here is a summary of...",The image shows a cellphone holder with a flex...
3,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0B93H178F,B0B93H178F,"This is a fantastic charger, especially if you...",5,"Shargeek USB C Charger 140W Wall Charger, Mult...","[-0.0699621961, 0.1709383726, -0.0337884836, -...",https://m.media-amazon.com/images/I/61R6UXrAKT...,https://m.media-amazon.com/images/I/61R6UXrAKT...,https://m.media-amazon.com/images/I/61R6UXrAKT...,"[21.9141178131, 16.1274204254, 0.0, 10.3457145...","[-0.50728655, -0.22111754, 0.44898894, -0.2970...","[-0.63115484, -0.17968105, 0.4211614, -0.10539...","[-0.053886086, 0.1409396, -0.025606845, -0.112...","[-0.05314668, 0.1396852, -0.016387641, -0.1113...","Based on the user review, here is a summary of...",The image shows a USB C charger with a power r...
4,AFERCDY2EFJKT7QUQ75GISNHTFOQ,B0BPXZWCBG,B0BPY47SY6,There never seems to be enough charging cords ...,4,"Charging Station for Multiple Devices,60W 12A ...","[-0.09403285380000001, 0.16169938450000002, -0...",https://m.media-amazon.com/images/I/51fqzioGKe...,https://m.media-amazon.com/images/I/51fqzioGKe...,https://m.media-amazon.com/images/I/51fqzioGKe...,"[0.0, 25.8072109222, 0.0, 35.9537582397, 0.0, ...","[-0.30125153, -0.10725977, 0.31016856, -0.1977...","[-0.442251, -0.18747163, 0.23647779, -0.066458...","[-0.06674541, 0.15736866, -0.014765534, -0.073...","[-0.08094199, 0.124280974, -0.008204151, -0.07...","Based on user reviews, it appears that users v...",The image shows a product packaging for a USB ...


In [12]:
df['user_id'].nunique()

4149

In [13]:
df['asin'].nunique()

4975

## Create Item Embedding

# User, Item label encoding
- 라벨 인코딩
- train, test 분리



In [20]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

In [21]:
# 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

# train과 test data를 분리
train, test = train_test_split(df, shuffle=True, test_size=0.2, random_state=42)
train, valid = train_test_split(train, shuffle=True, test_size=0.125, random_state=42)

# index reset
train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

In [22]:
# User ID
user_id_train = train['user_id'].to_numpy()
user_id_valid = valid['user_id'].to_numpy()
user_id_test = test['user_id'].to_numpy()

# Item ID
item_id_train = train['asin'].to_numpy()
item_id_valid = valid['asin'].to_numpy()
item_id_test = test['asin'].to_numpy()

# Image Set Mean by Item
image_feat_train = np.stack(train['VGG16_emb'].values).astype(np.float32)
image_feat_valid = np.stack(valid['VGG16_emb'].values).astype(np.float32)
image_feat_test = np.stack(test['VGG16_emb'].values).astype(np.float32)

# Text Feature Mean by Item
text_feat_train = np.stack(train['roberta_emb'].to_numpy()).astype(np.float32)
text_feat_valid = np.stack(valid['roberta_emb'].to_numpy()).astype(np.float32)
text_feat_test = np.stack(test['roberta_emb'].to_numpy()).astype(np.float32)

# rating
y_train = train['rating'].to_numpy()
y_valid = valid['rating'].to_numpy()
y_test = test['rating'].to_numpy()

# Create Embedding Matrix

In [23]:
# asin(item)별 groupby 후 평균
item_emb_df = df.groupby('asin').agg({
    'roberta_emb': lambda x: np.mean(np.stack(x), axis=0),
    'VGG16_emb': lambda x: np.mean(np.stack(x), axis=0)
}).reset_index()

item_emb_df.columns = ['asin', 'item_text_emb', 'item_image_emb']

# 데이터셋에 결과를 concat
df = df.merge(item_emb_df, on='asin', how='left')

# 결과 확인
item_emb_df.head()

,asin,item_text_emb,item_image_emb
0,0,"[-0.0960017045, 0.1450759818, -0.0343486977333...","[0.0, 25.880264918000005, 23.5852470398, 26.42..."
1,1,"[-0.09460095066666668, 0.15601724386666668, -0...","[3.4571736653666663, 32.10803349813333, 22.711..."
2,2,"[-0.10685905813333334, 0.13229569296666668, -0...","[5.8139998118, 23.27445062, 16.74244562786667,..."
3,3,"[-0.07609084493333333, 0.11230668426666666, -0...","[6.7305138906, 30.447373072299996, 2.550666133..."
4,4,"[-0.1066936155, 0.15352500973333336, -0.029108...","[12.963880221066667, 27.763807614633336, 18.12..."


In [28]:
# 필요한 배열로 변환
item_ids   = item_emb_df['asin'].to_numpy(dtype=np.int64)  # 라벨인코딩된 정수 ID
text_stack = np.stack(item_emb_df['item_text_emb'].to_numpy())   # (m, d_text)
img_stack  = np.stack(item_emb_df['item_image_emb'].to_numpy())  # (m, d_vis)

d_text_raw = text_stack.shape[1]
d_vis_raw  = img_stack.shape[1]

# 초기화(미등장 아이템 0벡터 처리)
item_text_mat  = np.zeros((n_items, d_text_raw), dtype=np.float32)
item_vis_mat = np.zeros((n_items, d_vis_raw),  dtype=np.float32)

# 해당 ID 위치에 채우기
item_text_mat[item_ids]  = text_stack.astype(np.float32)
item_vis_mat[item_ids] = img_stack.astype(np.float32)

print(item_text_mat.shape, item_vis_mat.shape)  # (n_items, d_text_raw), (n_items, d_vis_raw)

(4975, 768) (4975, 4096)


In [29]:
adj_norm_tf = build_tf_sparse_adj(user_id_train, item_id_train, n_users, n_items)

# MMGACL

In [114]:
# 4) GNN 인코더 + 대조학습 projector(MLP) + 점수기
#    - 외형은 Subclassing, 내부는 Functional/Layer 조합
#    - 주손실은 MSE, self.losses(부가 손실) 합산 유지
#    - 전역 L2는 AdamW(weight_decay)로 처리 (컴파일 시 설정)

class MMGACL_TF(Model):
    def __init__(self, n_users, n_items, d,
                 adj_norm_tf: tf.SparseTensor,
                 item_text_mat, item_vis_mat,
                 gcn_layers=2, gcn_dropout=0.0, use_residual=True,
                 d_text_raw=None, d_vis_raw=None,
                 projector_hidden=0,  # contrastive projector 크기(0이면 identity)
                 **kwargs):
        super().__init__(**kwargs)
        self.n_users = n_users
        self.n_items = n_items
        self.d = d

        # 아이템 원시 특징을 상수 텐서로 보관
        self.item_text_mat = tf.constant(item_text_mat, dtype=tf.float32)  # (n_items, d_text_raw)
        self.item_vis_mat  = tf.constant(item_vis_mat,  dtype=tf.float32)  # (n_items, d_vis_raw)

        # (1) 아이템 인코더: 모달 투영(제공함수) + 이중모달 어텐션 + 선형 사상
        assert d_text_raw is not None and d_vis_raw is not None
        self.item_encoder = build_item_encoder_with_bimodal_attention(d_text_raw, d_vis_raw, d)

        # (2) ID 임베딩
        self.user_emb = Embedding(n_users, d, embeddings_initializer="glorot_uniform", name="user_id_embedding")
        self.item_id_emb = Embedding(n_items, d, embeddings_initializer="glorot_uniform", name="item_id_embedding")

        # (3) 그래프 전파 스택
        self.gcn_stack = []
        for li in range(gcn_layers):
            self.gcn_stack.append(
                GraphProp(adj_norm_tf,
                          activation=(tf.nn.relu if li < gcn_layers - 1 else None),
                          dropout_rate=gcn_dropout,
                          use_residual=use_residual,
                          name=f"gcn_prop_{li+1}")
            )

        # (4) 대조학습 projector(옵션, 논문에서 GNN 이후 projector는 MLP)
        #     - 여기서는 기본 1~2층 정도로 구성 가능. 0이면 통과.
        self.proj_head = None
        if projector_hidden and projector_hidden > 0:
            z_in = Input(shape=(d,))
            z = Dense(projector_hidden, activation="relu", name="proj_hidden")(z_in)
            z = Dense(d, activation=None, name="proj_out")(z)
            self.proj_head = Model(z_in, z, name="ProjectorMLP")

        # (5) 점수기: 내적
        self.dot = tf.keras.layers.Dot(axes=-1, name="dot_score")

    def call(self, inputs, training=False):
        user_ids = tf.cast(inputs["user_id"], tf.int32)
        item_ids = tf.cast(inputs["item_id"], tf.int32)

        # a) 전 아이템에 대해 모달 융합 임베딩 계산
        item_modal_all = self.item_encoder([self.item_text_mat, self.item_vis_mat], training=training)  # (n_items, d)

        # b) ID 임베딩
        U0 = self.user_emb(tf.range(self.n_users))    # (n_users, d)
        I0 = self.item_id_emb(tf.range(self.n_items)) # (n_items, d)

        # c) 아이템 초기 임베딩: ID + 모달
        I_init = I0 + item_modal_all                  # (n_items, d)

        # d) GCN 전파
        X = tf.concat([U0, I_init], axis=0)           # (n_users+n_items, d)
        for gcn in self.gcn_stack:
            X = gcn(X, training=training)
        U = X[:self.n_users, :]
        I = X[self.n_items * 0 + self.n_users:, :]

        # e) projector (옵션)
        if self.proj_head is not None:
            U = self.proj_head(U, training=training)
            I = self.proj_head(I, training=training)

        # f) 배치 점수
        u_vec = tf.gather(U, user_ids)                # (B, d)
        i_vec = tf.gather(I, item_ids)                # (B, d)
        y_pred = self.dot([u_vec, i_vec])             # (B, 1)
        return tf.squeeze(y_pred, axis=-1)            # (B,)

    @tf.function
    def train_step(self, data):
        if isinstance(data, (tuple, list)):
            inputs, y_true = data
        else:
            inputs = {k: data[k] for k in ["user_id", "item_id"]}
            y_true = data["y_true"]

        y_true = tf.cast(y_true, tf.float32)

        with tf.GradientTape() as tape:
            y_pred = self(inputs, training=True)

            # ✅ Rank 고정
            y_pred = tf.reshape(y_pred, [-1])
            y_true = tf.reshape(y_true, [-1])

            loss = self.compiled_loss(y_true, y_pred)
            reg_loss = tf.add_n(self.losses) if self.losses else 0.0
            total_loss = loss + reg_loss

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        for metric in self.metrics:
            metric.update_state(y_true, y_pred)

        results = {m.name: m.result() for m in self.metrics}
        results.update({"loss": total_loss})
        return results

In [115]:
# 하이퍼 파라미터
# 0이면 projector 비활성
d=64
gcn_layers=1
gcn_dropout=0.1
use_residual=True
projector_hidden=128
lr=1e-1
weight_decay=1e-4

In [116]:
model = MMGACL_TF(
        n_users=n_users, n_items=n_items, d=d,
        adj_norm_tf=adj_norm_tf,
        item_text_mat=item_text_mat,
        item_vis_mat=item_vis_mat,
        gcn_layers=gcn_layers,
        gcn_dropout=gcn_dropout,
        use_residual=use_residual,
        d_text_raw=768,
        d_vis_raw=4096,
        projector_hidden=projector_hidden,
    )

model.summary()

# model.layers[4].summary()

Model: "mmgacl_tf_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ItemEncoder_BiAttn (Functional) │ (None, 64)             │       323,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ user_id_embedding (Embedding)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ item_id_embedding (Embedding)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gcn_prop_1 (GraphProp)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ProjectorMLP (Functional)       │ (None, 64)             │        16,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dot_score (Dot)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 340,352 (1.30 MB)

 Trainable params: 340,352 (1.30 MB)

 Non-trainable params: 0 (0.00 B)

In [117]:
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=['mean_absolute_error', 'mean_squared_error'],
    jit_compile=False
)

In [118]:
# Configuration of EarlyStopping
early_stopping = EarlyStopping(
    monitor ='val_loss',  # Monitor model's validation loss
    min_delta = 0.01,     # Minimum change to qualify as an improvement
    patience = 5,         # Stop after 5 epochs without improvement
    verbose = 1,           # Print messages
    mode = 'min',          # Stop training when the quantity monitored has stopped decreasing
    restore_best_weights = True  # Restore model weights from the epoch with the best value of the monitored quantity
)

In [119]:
train_inputs = {
    "user_id":    user_id_train,      # int32, shape (N,)
    "item_id":    item_id_train,      # int32, shape (N,)

}
valid_inputs = {
    "user_id":    user_id_valid,
    "item_id":    item_id_valid,
}

history = model.fit(
    x=train_inputs,
    y=y_train,
    validation_data=(valid_inputs, y_valid),
    epochs=100,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1,
)

Epoch 1/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - mean_absolute_error: nan - mean_squared_error: nan - loss: nan - val_loss: nan - val_mean_absolute_error: nan - val_mean_squared_error: nan
Epoch 2/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - mean_absolute_error: nan - mean_squared_error: nan - loss: nan - val_loss: nan - val_mean_absolute_error: nan - val_mean_squared_error: nan
Epoch 3/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - mean_absolute_error: nan - mean_squared_error: nan - loss: nan - val_loss: nan - val_mean_absolute_error: nan - val_mean_squared_error: nan
Epoch 4/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - mean_absolute_error: nan - mean_squared_error: nan - loss: nan - val_loss: nan - val_mean_absolute_error: nan - val_mean_squared_error: nan
Epoch 5/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - mean_absolute_error: nan - mean_squared_error: nan - loss: nan - val_loss: nan - val_mean_absolute_error: nan - val_mean_squared_error: nan
Epoch 6/100
129/129 ━━━━━

In [107]:
# 1) 테스트 입력 준비 (dict)
test_inputs = {
    "user_id":    user_id_test.astype(np.int32),
    "item_id":    item_id_test.astype(np.int32),
}

# 2) 예측 (회귀 점수)
y_pred = model.predict(test_inputs)
y_pred = np.asarray(y_pred, dtype=np.float32).ravel()

# MAE
MAE = mean_absolute_error(y_test, y_pred)
print(f"MAE: {MAE:.4f}")

# MSE
MSE = mean_squared_error(y_test, y_pred)
print(f"MSE: {MSE:.4f}")

# RMSE
RMSE = np.sqrt(MSE)
print(f"RMSE: {RMSE:.4f}")

# MAPE
MAPE = mean_absolute_percentage_error(y_test, y_pred) * 100

print(f"MAPE: {MAPE:.4f}%")

147/147 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
MAE: 0.6112
MSE: 0.6495
RMSE: 0.8059
MAPE: 18.5079%
